# 02 - Train-Only Feature Exploration

This notebook performs predictive exploratory data analysis on the rich engineered feature table for the training split only. It uses `data/processed/features_train.csv`, `data/processed/feature_manifest.csv`, and the training labels carried in the train feature table.


## EDA Contract

This notebook is allowed to form expectations about feature behavior, feature families, rolling/context features, subject-normalized features, and redundancy. It should not inspect validation or test feature tables, labels, metrics, predictions, or model outputs.

The output should be treated as hypothesis-generating evidence for later ablations and model interpretation, not as final model-selection evidence.


## Setup


In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import TARGET_LABELS
from src.features.build_features import FEATURE_ID_COLUMNS

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 160)

TRAIN_FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features_train.csv"
FEATURE_MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "feature_manifest.csv"

STAGE_ORDER = [label for label in TARGET_LABELS]
RANDOM_STATE = 42
MUTUAL_INFO_SAMPLE_SIZE = 20000
TOP_N = 25
HIGH_CORRELATION_THRESHOLD = 0.95

print(f"Project root: {PROJECT_ROOT}")


## Load Train Artifacts


In [ ]:
required_paths = {
    "training feature table": TRAIN_FEATURES_PATH,
    "feature manifest": FEATURE_MANIFEST_PATH,
}
missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required artifact(s): "
        + ", ".join(missing_paths)
        + ". Run scripts/build_features.py first."
    )

features_train = pd.read_csv(TRAIN_FEATURES_PATH, dtype={"participant_id": str})
feature_manifest = pd.read_csv(FEATURE_MANIFEST_PATH)

missing_id_columns = sorted(set(FEATURE_ID_COLUMNS) - set(features_train.columns))
if missing_id_columns:
    raise ValueError(f"Training feature table is missing required columns: {missing_id_columns}")

train_splits = set(features_train["split"].astype(str).str.strip())
if train_splits != {"train"}:
    raise ValueError(
        "This notebook must use the training feature table only; "
        f"found split values: {sorted(train_splits)}"
    )

observed_labels = set(features_train["label"].astype(str))
unexpected_labels = sorted(observed_labels - set(STAGE_ORDER))
if unexpected_labels:
    raise ValueError(f"Unexpected label values in training table: {unexpected_labels}")

feature_columns = [
    column for column in features_train.columns if column not in FEATURE_ID_COLUMNS
]
manifest_features = set(feature_manifest["feature"])
table_features = set(feature_columns)
if table_features != manifest_features:
    missing_from_manifest = sorted(table_features - manifest_features)
    missing_from_table = sorted(manifest_features - table_features)
    raise ValueError(
        "Feature manifest does not match the training table. "
        f"Missing from manifest: {missing_from_manifest[:10]}; "
        f"missing from table: {missing_from_table[:10]}"
    )

feature_manifest = feature_manifest.set_index("feature").loc[feature_columns].reset_index()
numeric_features = features_train[feature_columns].apply(pd.to_numeric, errors="coerce")

display(
    Markdown(
        f"Loaded **{len(features_train):,}** training epochs from "
        f"**{features_train['participant_id'].nunique():,}** participants with "
        f"**{len(feature_columns):,}** engineered features."
    )
)
display(features_train[FEATURE_ID_COLUMNS].head())


## Training Class Balance


In [ ]:
class_counts = (
    features_train["label"]
    .value_counts()
    .reindex(STAGE_ORDER)
    .rename_axis("label")
    .to_frame("n_epochs")
)
class_counts["share"] = class_counts["n_epochs"] / class_counts["n_epochs"].sum()

participant_label_counts = (
    features_train.groupby(["participant_id", "label"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=STAGE_ORDER, fill_value=0)
)
participant_label_share = participant_label_counts.div(
    participant_label_counts.sum(axis=1), axis=0
)

display(class_counts.style.format({"share": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(
    data=class_counts.reset_index(),
    x="label",
    y="n_epochs",
    order=STAGE_ORDER,
    ax=axes[0],
)
axes[0].set_title("Training epochs by sleep stage")
axes[0].set_xlabel("")
axes[0].set_ylabel("Epochs")

participant_label_share[STAGE_ORDER].plot(
    kind="box",
    ax=axes[1],
    showfliers=False,
)
axes[1].set_title("Participant-level label-share variability")
axes[1].set_xlabel("")
axes[1].set_ylabel("Within-participant label share")
fig.tight_layout()
plt.show()


## Feature Inventory and Coverage


In [ ]:
feature_inventory = (
    feature_manifest.groupby(["feature_family", "signal_group"], observed=True)
    .size()
    .rename("n_features")
    .reset_index()
    .pivot(index="feature_family", columns="signal_group", values="n_features")
    .fillna(0)
    .astype(int)
)

coverage = pd.DataFrame(
    {
        "feature": feature_columns,
        "finite_fraction": np.isfinite(numeric_features.to_numpy()).mean(axis=0),
        "missing_fraction": numeric_features.isna().mean(axis=0).to_numpy(),
        "n_unique": numeric_features.nunique(dropna=True).to_numpy(),
    }
).merge(feature_manifest, on="feature", how="left")
coverage["is_usable_univariate"] = (
    coverage["finite_fraction"].gt(0) & coverage["n_unique"].gt(1)
)

coverage_by_family = (
    coverage.groupby(["feature_family", "signal_group"], observed=True)
    .agg(
        n_features=("feature", "size"),
        usable_features=("is_usable_univariate", "sum"),
        median_finite_fraction=("finite_fraction", "median"),
        max_missing_fraction=("missing_fraction", "max"),
    )
    .reset_index()
    .sort_values(["feature_family", "signal_group"])
)

display(feature_inventory)
display(
    coverage_by_family.style.format(
        {
            "median_finite_fraction": "{:.1%}",
            "max_missing_fraction": "{:.1%}",
        }
    )
)

low_coverage = coverage.sort_values(["finite_fraction", "n_unique"]).head(20)
display(Markdown("Lowest-coverage or least-variable training features:"))
display(
    low_coverage[
        [
            "feature",
            "feature_family",
            "signal_group",
            "finite_fraction",
            "missing_fraction",
            "n_unique",
        ]
    ].style.format(
        {"finite_fraction": "{:.1%}", "missing_fraction": "{:.1%}"}
    )
)


## Univariate Class-Separation Checks


In [ ]:
def stratified_sample(frame: pd.DataFrame, n_rows: int, random_state: int) -> pd.DataFrame:
    if len(frame) <= n_rows:
        return frame
    label_counts = frame["label"].value_counts()
    fractions = label_counts / label_counts.sum()
    allocations = (fractions * n_rows).round().astype(int).clip(lower=1)
    while allocations.sum() > n_rows:
        allocations.loc[allocations.idxmax()] -= 1
    while allocations.sum() < n_rows:
        allocations.loc[(label_counts - allocations).idxmax()] += 1
    sampled = []
    for label, n_label in allocations.items():
        label_rows = frame[frame["label"].eq(label)]
        sampled.append(label_rows.sample(n=min(n_label, len(label_rows)), random_state=random_state))
    return pd.concat(sampled).sample(frac=1, random_state=random_state)


def eta_squared(feature_frame: pd.DataFrame, labels: pd.Series, columns: list[str]) -> pd.Series:
    values = feature_frame[columns]
    rows = {}
    for column in columns:
        series = values[column]
        valid = series.notna() & labels.notna()
        if valid.sum() < 2:
            rows[column] = np.nan
            continue
        y = labels[valid]
        x = series[valid]
        overall_mean = x.mean()
        ss_total = ((x - overall_mean) ** 2).sum()
        if ss_total == 0:
            rows[column] = np.nan
            continue
        ss_between = 0.0
        for label, group in x.groupby(y, observed=True):
            ss_between += len(group) * (group.mean() - overall_mean) ** 2
        rows[column] = ss_between / ss_total
    return pd.Series(rows, name="eta_squared")


def median_gap_by_iqr(feature_frame: pd.DataFrame, labels: pd.Series, columns: list[str]) -> pd.Series:
    rows = {}
    for column in columns:
        series = feature_frame[column]
        valid = series.notna() & labels.notna()
        if valid.sum() < 2:
            rows[column] = np.nan
            continue
        medians = series[valid].groupby(labels[valid], observed=True).median()
        iqr = series[valid].quantile(0.75) - series[valid].quantile(0.25)
        rows[column] = np.nan if iqr == 0 else (medians.max() - medians.min()) / iqr
    return pd.Series(rows, name="median_gap_iqr")


usable_features = coverage.loc[coverage["is_usable_univariate"], "feature"].tolist()
y = features_train["label"].astype(str)

imputer = SimpleImputer(strategy="median")
X_anova = imputer.fit_transform(numeric_features[usable_features])
anova_f, anova_p = f_classif(X_anova, y)

mi_frame = stratified_sample(
    features_train[["label", *usable_features]],
    MUTUAL_INFO_SAMPLE_SIZE,
    RANDOM_STATE,
)
X_mi = SimpleImputer(strategy="median").fit_transform(
    mi_frame[usable_features].apply(pd.to_numeric, errors="coerce")
)
y_mi = LabelEncoder().fit_transform(mi_frame["label"].astype(str))
mutual_info = mutual_info_classif(
    X_mi,
    y_mi,
    discrete_features=False,
    random_state=RANDOM_STATE,
)

univariate_scores = (
    pd.DataFrame(
        {
            "feature": usable_features,
            "anova_f": anova_f,
            "anova_p": anova_p,
            "mutual_info": mutual_info,
        }
    )
    .merge(eta_squared(numeric_features, y, usable_features).reset_index().rename(columns={"index": "feature"}), on="feature")
    .merge(median_gap_by_iqr(numeric_features, y, usable_features).reset_index().rename(columns={"index": "feature"}), on="feature")
    .merge(feature_manifest, on="feature", how="left")
)

univariate_scores["anova_rank"] = univariate_scores["anova_f"].rank(
    method="min", ascending=False
)
univariate_scores["mutual_info_rank"] = univariate_scores["mutual_info"].rank(
    method="min", ascending=False
)
univariate_scores["eta_rank"] = univariate_scores["eta_squared"].rank(
    method="min", ascending=False
)
univariate_scores["combined_rank"] = (
    univariate_scores[["anova_rank", "mutual_info_rank", "eta_rank"]]
    .mean(axis=1)
    .rank(method="min")
)
univariate_scores = univariate_scores.sort_values(
    ["combined_rank", "anova_rank", "mutual_info_rank"]
)

top_univariate = univariate_scores.head(TOP_N)
display(
    top_univariate[
        [
            "feature",
            "feature_family",
            "signal_group",
            "source_signal",
            "anova_f",
            "mutual_info",
            "eta_squared",
            "median_gap_iqr",
            "combined_rank",
        ]
    ].style.format(
        {
            "anova_f": "{:.2f}",
            "mutual_info": "{:.4f}",
            "eta_squared": "{:.4f}",
            "median_gap_iqr": "{:.2f}",
            "combined_rank": "{:.0f}",
        }
    )
)


## Top Feature Distributions by Sleep Stage


In [ ]:
def winsorized_long_frame(
    frame: pd.DataFrame,
    columns: list[str],
    label_col: str = "label",
    lower: float = 0.01,
    upper: float = 0.99,
) -> pd.DataFrame:
    pieces = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors="coerce")
        lo, hi = values.quantile([lower, upper])
        if pd.notna(lo) and pd.notna(hi) and lo < hi:
            values = values.clip(lo, hi)
        pieces.append(
            pd.DataFrame(
                {
                    "label": frame[label_col].astype(str),
                    "feature": column,
                    "value": values,
                }
            )
        )
    return pd.concat(pieces, ignore_index=True)


plot_features = top_univariate["feature"].head(8).tolist()
distribution_frame = winsorized_long_frame(features_train, plot_features)

g = sns.catplot(
    data=distribution_frame,
    x="label",
    y="value",
    col="feature",
    col_wrap=2,
    kind="box",
    order=STAGE_ORDER,
    sharey=False,
    showfliers=False,
    height=3.0,
    aspect=1.45,
)
g.set_axis_labels("", "Winsorized feature value")
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

class_summary_rows = []
for feature in plot_features:
    summary = (
        features_train.assign(value=pd.to_numeric(features_train[feature], errors="coerce"))
        .groupby("label", observed=True)["value"]
        .agg(["median", "mean", "std"])
        .reindex(STAGE_ORDER)
    )
    summary["feature"] = feature
    class_summary_rows.append(summary.reset_index())
class_summary = pd.concat(class_summary_rows, ignore_index=True)
display(
    class_summary[
        ["feature", "label", "median", "mean", "std"]
    ].style.format({"median": "{:.4g}", "mean": "{:.4g}", "std": "{:.4g}"})
)


## Signal-Family Expectations


In [ ]:
family_summary = (
    univariate_scores.groupby(["signal_group", "feature_family"], observed=True)
    .agg(
        n_features=("feature", "size"),
        best_combined_rank=("combined_rank", "min"),
        median_combined_rank=("combined_rank", "median"),
        median_anova_f=("anova_f", "median"),
        median_mutual_info=("mutual_info", "median"),
        top_25_features=("combined_rank", lambda s: int((s <= TOP_N).sum())),
    )
    .reset_index()
    .sort_values(["best_combined_rank", "median_combined_rank"])
)

signal_summary = (
    univariate_scores.groupby("signal_group", observed=True)
    .agg(
        n_features=("feature", "size"),
        best_combined_rank=("combined_rank", "min"),
        median_combined_rank=("combined_rank", "median"),
        top_25_features=("combined_rank", lambda s: int((s <= TOP_N).sum())),
    )
    .sort_values(["best_combined_rank", "median_combined_rank"])
)

display(signal_summary.style.format({"best_combined_rank": "{:.0f}", "median_combined_rank": "{:.1f}"}))
display(
    family_summary.style.format(
        {
            "best_combined_rank": "{:.0f}",
            "median_combined_rank": "{:.1f}",
            "median_anova_f": "{:.2f}",
            "median_mutual_info": "{:.4f}",
        }
    )
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=signal_summary.reset_index(),
    x="signal_group",
    y="top_25_features",
    order=signal_summary.index.tolist(),
    ax=ax,
)
ax.set_title(f"Signal groups represented in the top {TOP_N} univariate features")
ax.set_xlabel("")
ax.set_ylabel(f"Top {TOP_N} feature count")
plt.tight_layout()
plt.show()


## Rolling and Context Features


In [ ]:
ROLLING_PATTERN = re.compile(r"(.+)_roll(\d+)_(mean|std)$")


def rolling_source(feature: str) -> str | None:
    match = ROLLING_PATTERN.match(feature)
    return match.group(1) if match else None


def rolling_window(feature: str) -> int | None:
    match = ROLLING_PATTERN.match(feature)
    return int(match.group(2)) if match else None


rolling_scores = univariate_scores[
    univariate_scores["feature_family"].eq("rolling_context")
].copy()
rolling_scores["source_feature"] = rolling_scores["feature"].map(rolling_source)
rolling_scores["window_epochs"] = rolling_scores["feature"].map(rolling_window)

base_scores = univariate_scores[
    ["feature", "combined_rank", "anova_f", "mutual_info", "eta_squared"]
].rename(
    columns={
        "feature": "source_feature",
        "combined_rank": "source_combined_rank",
        "anova_f": "source_anova_f",
        "mutual_info": "source_mutual_info",
        "eta_squared": "source_eta_squared",
    }
)
rolling_comparison = rolling_scores.merge(base_scores, on="source_feature", how="left")
rolling_comparison["rank_delta_vs_source"] = (
    rolling_comparison["combined_rank"] - rolling_comparison["source_combined_rank"]
)

display(
    rolling_comparison.sort_values(["combined_rank", "rank_delta_vs_source"])
    .head(25)[
        [
            "feature",
            "source_feature",
            "window_epochs",
            "signal_group",
            "combined_rank",
            "source_combined_rank",
            "rank_delta_vs_source",
            "anova_f",
            "mutual_info",
            "eta_squared",
        ]
    ]
    .style.format(
        {
            "combined_rank": "{:.0f}",
            "source_combined_rank": "{:.0f}",
            "rank_delta_vs_source": "{:+.0f}",
            "anova_f": "{:.2f}",
            "mutual_info": "{:.4f}",
            "eta_squared": "{:.4f}",
        }
    )
)

rolling_window_summary = (
    rolling_comparison.groupby(["window_epochs", "signal_group"], observed=True)
    .agg(
        n_features=("feature", "size"),
        best_combined_rank=("combined_rank", "min"),
        median_combined_rank=("combined_rank", "median"),
        median_rank_delta_vs_source=("rank_delta_vs_source", "median"),
    )
    .reset_index()
    .sort_values(["window_epochs", "best_combined_rank"])
)
display(
    rolling_window_summary.style.format(
        {
            "best_combined_rank": "{:.0f}",
            "median_combined_rank": "{:.1f}",
            "median_rank_delta_vs_source": "{:+.1f}",
        }
    )
)


## Subject-Normalized Features


In [ ]:
normalized_scores = univariate_scores[
    univariate_scores["feature_family"].eq("whole_night_subject_normalized")
].copy()
normalized_scores["source_feature"] = normalized_scores["feature"].str.removesuffix(
    "_participant_z"
)

raw_scores = univariate_scores[
    ["feature", "combined_rank", "anova_f", "mutual_info", "eta_squared"]
].rename(
    columns={
        "feature": "source_feature",
        "combined_rank": "raw_combined_rank",
        "anova_f": "raw_anova_f",
        "mutual_info": "raw_mutual_info",
        "eta_squared": "raw_eta_squared",
    }
)

normalized_comparison = normalized_scores.merge(raw_scores, on="source_feature", how="inner")
normalized_comparison["rank_delta_vs_raw"] = (
    normalized_comparison["combined_rank"] - normalized_comparison["raw_combined_rank"]
)

variability_rows = []
for row in normalized_comparison.itertuples(index=False):
    raw_feature = row.source_feature
    normalized_feature = row.feature
    raw_participant_means = numeric_features.groupby(features_train["participant_id"])[raw_feature].mean()
    normalized_participant_means = numeric_features.groupby(features_train["participant_id"])[normalized_feature].mean()
    raw_between_sd = raw_participant_means.std()
    normalized_between_sd = normalized_participant_means.std()
    variability_rows.append(
        {
            "feature": normalized_feature,
            "source_feature": raw_feature,
            "raw_between_participant_mean_sd": raw_between_sd,
            "normalized_between_participant_mean_sd": normalized_between_sd,
            "between_participant_mean_sd_ratio": (
                normalized_between_sd / raw_between_sd
                if pd.notna(raw_between_sd) and raw_between_sd != 0
                else np.nan
            ),
        }
    )

normalized_variability = pd.DataFrame(variability_rows)
normalized_comparison = normalized_comparison.merge(
    normalized_variability,
    on=["feature", "source_feature"],
    how="left",
)

display(
    normalized_comparison.sort_values(["combined_rank", "rank_delta_vs_raw"])
    .head(25)[
        [
            "feature",
            "source_feature",
            "signal_group",
            "combined_rank",
            "raw_combined_rank",
            "rank_delta_vs_raw",
            "between_participant_mean_sd_ratio",
            "anova_f",
            "raw_anova_f",
            "mutual_info",
            "raw_mutual_info",
        ]
    ]
    .style.format(
        {
            "combined_rank": "{:.0f}",
            "raw_combined_rank": "{:.0f}",
            "rank_delta_vs_raw": "{:+.0f}",
            "between_participant_mean_sd_ratio": "{:.4f}",
            "anova_f": "{:.2f}",
            "raw_anova_f": "{:.2f}",
            "mutual_info": "{:.4f}",
            "raw_mutual_info": "{:.4f}",
        }
    )
)

normalization_summary = (
    normalized_comparison.groupby("signal_group", observed=True)
    .agg(
        n_pairs=("feature", "size"),
        normalized_better_than_raw=("rank_delta_vs_raw", lambda s: int((s < 0).sum())),
        median_rank_delta_vs_raw=("rank_delta_vs_raw", "median"),
        median_between_participant_mean_sd_ratio=(
            "between_participant_mean_sd_ratio",
            "median",
        ),
    )
    .sort_values("median_rank_delta_vs_raw")
)
display(
    normalization_summary.style.format(
        {
            "median_rank_delta_vs_raw": "{:+.1f}",
            "median_between_participant_mean_sd_ratio": "{:.4f}",
        }
    )
)


## Redundancy and Correlation


In [ ]:
correlation_features = usable_features
correlation_matrix = numeric_features[correlation_features].corr(method="pearson").abs()
upper_mask = np.triu(np.ones(correlation_matrix.shape, dtype=bool), k=1)
high_corr_pairs = (
    correlation_matrix.where(upper_mask)
    .stack()
    .rename("abs_pearson_corr")
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
    .query("abs_pearson_corr >= @HIGH_CORRELATION_THRESHOLD")
    .sort_values("abs_pearson_corr", ascending=False)
)

ranked_lookup = univariate_scores.set_index("feature")[
    ["combined_rank", "feature_family", "signal_group", "source_signal"]
]
high_corr_pairs = high_corr_pairs.join(ranked_lookup, on="feature_a").rename(
    columns={
        "combined_rank": "feature_a_rank",
        "feature_family": "feature_a_family",
        "signal_group": "feature_a_group",
        "source_signal": "feature_a_source",
    }
)
high_corr_pairs = high_corr_pairs.join(ranked_lookup, on="feature_b").rename(
    columns={
        "combined_rank": "feature_b_rank",
        "feature_family": "feature_b_family",
        "signal_group": "feature_b_group",
        "source_signal": "feature_b_source",
    }
)

display(
    Markdown(
        f"Found **{len(high_corr_pairs):,}** feature pairs with absolute Pearson "
        f"correlation >= **{HIGH_CORRELATION_THRESHOLD:.2f}** in the training table."
    )
)
display(
    high_corr_pairs.head(40)[
        [
            "feature_a",
            "feature_b",
            "abs_pearson_corr",
            "feature_a_family",
            "feature_b_family",
            "feature_a_group",
            "feature_b_group",
            "feature_a_rank",
            "feature_b_rank",
        ]
    ].style.format(
        {
            "abs_pearson_corr": "{:.3f}",
            "feature_a_rank": "{:.0f}",
            "feature_b_rank": "{:.0f}",
        }
    )
)

correlation_family_summary = (
    high_corr_pairs.assign(
        family_pair=lambda df: np.where(
            df["feature_a_family"].le(df["feature_b_family"]),
            df["feature_a_family"] + " | " + df["feature_b_family"],
            df["feature_b_family"] + " | " + df["feature_a_family"],
        ),
        group_pair=lambda df: np.where(
            df["feature_a_group"].le(df["feature_b_group"]),
            df["feature_a_group"] + " | " + df["feature_b_group"],
            df["feature_b_group"] + " | " + df["feature_a_group"],
        ),
    )
    .groupby(["family_pair", "group_pair"], observed=True)
    .size()
    .rename("n_high_corr_pairs")
    .reset_index()
    .sort_values("n_high_corr_pairs", ascending=False)
)
display(correlation_family_summary.head(30))


## Train-Only EDA Expectations


In [ ]:
leading_features = ", ".join(top_univariate["feature"].head(8).tolist())
leading_signal_groups = ", ".join(signal_summary.head(3).index.tolist())

rolling_top = rolling_comparison.sort_values("combined_rank").head(5)["feature"].tolist()
normalized_top = (
    normalized_comparison.sort_values("combined_rank").head(5)["feature"].tolist()
)
redundancy_note = (
    f"{len(high_corr_pairs):,} pairs exceed |r| >= {HIGH_CORRELATION_THRESHOLD:.2f}"
)

display(
    Markdown(
        f"""
        ### Current expectations from training-only feature EDA

        - Strongest univariate separators to inspect first: {leading_features}.
        - Signal groups most represented near the top of the univariate rankings: {leading_signal_groups}.
        - Rolling/context features worth tracking in ablations: {", ".join(rolling_top)}.
        - Subject-normalized features worth comparing against raw counterparts: {", ".join(normalized_top)}.
        - Redundancy is expected to matter: {redundancy_note}.

        These expectations come only from `features_train.csv` and should be checked later through train-only cross-validation and validation-set model comparison, with the test split still held out.
        """
    )
)


## Reproducibility Notes

This notebook intentionally does not load `features_val.csv`, `features_test.csv`, validation/test labels, model predictions, or metrics. Mutual information uses a stratified training-only sample to keep runtime predictable; increase `MUTUAL_INFO_SAMPLE_SIZE` in the setup cell if you want the full training table used for that score.
